# Embeddings

This lab is designed to help you solidify your understanding of embeddings by applying them to tasks like semantic similarity, clustering, and building a semantic search system.

### Tasks:
- Task 1: Semantic Similarity Comparison
- Task 2: Document Clustering
- Task 3: Enhance the Semantic Search System


## Task 1: Semantic Similarity Comparison
### Objective:
Compare semantic similarity between pairs of sentences using cosine similarity and embeddings.

### Steps:
1. Load a pre-trained Sentence Transformer model.
2. Encode the sentence pairs.
3. Compute cosine similarity for each pair.

### Dataset:
- "A dog is playing in the park." vs. "A dog is running in a field."
- "I love pizza." vs. "I enjoy ice cream."
- "What is AI?" vs. "How does a computer learn?"


In [1]:
from sentence_transformers import SentenceTransformer  # Imports SentenceTransformer so we can load a public embedding model from Hugging Face.
from sklearn.metrics.pairwise import cosine_similarity  # Imports cosine_similarity so sentence vectors can be compared numerically.

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  # Loads the public MiniLM sentence embedding model.
sentence_pairs = [  # Creates the sentence pairs that the lab asks us to compare.
    ("A dog is playing in the park.", "A dog is running in a field."),  # Stores a pair about similar dog activities.
    ("I love pizza.", "I enjoy ice cream."),  # Stores a pair about different foods and preferences.
    ("What is AI?", "How does a computer learn?"),  # Stores a pair about related technology questions.
]  # Closes the list of sentence pairs.
for first_sentence, second_sentence in sentence_pairs:  # Loops through every sentence pair.
    pair_embeddings = model.encode([first_sentence, second_sentence])  # Encodes both sentences into dense vectors.
    similarity_score = cosine_similarity([pair_embeddings[0]], [pair_embeddings[1]])[0][0]  # Computes cosine similarity between the two vectors.
    print(f"{similarity_score:.3f} -> {first_sentence} | {second_sentence}")# Reports the similarity score beside the sentence pair.


c:\tf310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10302.47it/s]


0.522 -> A dog is playing in the park. | A dog is running in a field.
0.528 -> I love pizza. | I enjoy ice cream.
0.319 -> What is AI? | How does a computer learn?


### Questions:
- Which sentence pairs are the most semantically similar? Why?
- Can you think of cases where cosine similarity might fail to capture true semantic meaning?


## Task 2: Document Clustering
### Objective:
Cluster a set of text documents into similar groups based on their embeddings.

### Steps:
1. Encode the documents using Sentence Transformers.
2. Use KMeans clustering to group the documents.
3. Analyze the clusters for semantic meaning.

In [2]:
from sklearn.cluster import KMeans  # Imports KMeans so documents can be grouped by embedding similarity.

documents = [  # Creates the short document collection used for clustering.
    "What is the capital of France?",  # Adds a geography question.
    "How do I bake a chocolate cake?",  # Adds a cooking question.
    "What is the distance between Earth and Mars?",  # Adds a space/science question.
    "How do I change a flat tire on a car?",  # Adds a car repair question.
    "What is the best way to learn Python?",  # Adds a programming learning question.
    "How do I fix a leaky faucet?",  # Adds a home repair question.
]  # Closes the document list.
document_embeddings = model.encode(documents)  # Encodes every document into an embedding vector.
print(document_embeddings.shape)# Confirms the current array or table dimensions.


(6, 384)


In [3]:
num_clusters = 3  # Chooses three clusters because the dataset mixes knowledge, practical how-to, and technical/science questions.
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init="auto")  # Creates a reproducible KMeans clustering model.
cluster_labels = kmeans.fit_predict(document_embeddings)  # Fits KMeans on the document embeddings and returns a cluster label per document.


In [4]:
for document, cluster_label in zip(documents, cluster_labels):  # Loops through each document and its assigned cluster label.
    print(f"Cluster {cluster_label}: {document}")# Reports each document cluster assignment.


Cluster 2: What is the capital of France?
Cluster 0: How do I bake a chocolate cake?
Cluster 0: What is the distance between Earth and Mars?
Cluster 1: How do I change a flat tire on a car?
Cluster 2: What is the best way to learn Python?
Cluster 1: How do I fix a leaky faucet?


### Questions:
- How many clusters make the most sense? Why?
- Examine the documents in each cluster. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?
- Try this exercise with a larger dataset of your choice

## Task 3: Semantic Search System
### Objective:
Create a semantic search engine:
A user provides a query and you search the dataset for semantically relevant documents to return. Return the top 5 results.

### Dataset:
- Use the following set of documents:
    - "What is the capital of France?"
    - "How do I bake a chocolate cake?"
    - "What is the distance between Earth and Mars?"
    - "How do I change a flat tire on a car?"
    - "What is the best way to learn Python?"
    - "How do I fix a leaky faucet?"
    - "What are the best travel destinations in Europe?"
    - "How do I set up a local server?"
    - "What is quantum computing?"
    - "How do I build a mobile app?"


In [5]:
import numpy as np  # Imports NumPy for sorting similarity scores.

documents = [  # Creates the document collection used by the semantic search system.
    "What is the capital of France?",  # Adds a geography question.
    "How do I bake a chocolate cake?",  # Adds a cooking question.
    "What is the distance between Earth and Mars?",  # Adds a space/science question.
    "How do I change a flat tire on a car?",  # Adds a vehicle repair question.
    "What is the best way to learn Python?",  # Adds a programming question.
    "How do I fix a leaky faucet?",  # Adds a home repair question.
    "What are the best travel destinations in Europe?",  # Adds a travel question.
    "How do I set up a local server?",  # Adds a technical setup question.
    "What is quantum computing?",  # Adds a computing/science question.
    "How do I build a mobile app?",  # Adds a software development question.
]  # Closes the document list.
doc_embeddings = model.encode(documents)  # Encodes the full search corpus into document embeddings.
print(doc_embeddings.shape)# Confirms the current array or table dimensions.


(10, 384)


In [6]:
def semantic_search(query, documents, doc_embeddings, top_n=5):  # Defines a semantic search helper that returns the most relevant documents.
    query_embedding = model.encode([query])  # Encodes the user query into the same embedding space as the documents.
    similarity_scores = cosine_similarity(query_embedding, doc_embeddings)[0]  # Computes similarity between the query and every document.
    top_indices = np.argsort(similarity_scores)[::-1][:top_n]  # Sorts scores from highest to lowest and keeps the top requested results.
    results = []  # Creates an empty list for the ranked search results.
    for rank, document_index in enumerate(top_indices, start=1):  # Loops through the best document indices in ranked order.
        result = {"rank": rank, "score": similarity_scores[document_index], "document": documents[document_index]}  # Builds one result record.
        results.append(result)  # Adds the result record to the output list.
        print(f"{rank}. ({similarity_scores[document_index]:.3f}) {documents[document_index]}")# Reports the similarity score beside the sentence pair.
    return results  # Returns the result list for later use.


In [7]:
query = "Explain programming languages."  # Defines a test query about programming.
semantic_search(query, documents, doc_embeddings)  # Runs semantic search and prints the top matching documents.


1. (0.435) What is quantum computing?
2. (0.319) What is the best way to learn Python?
3. (0.110) How do I build a mobile app?
4. (0.091) How do I set up a local server?
5. (0.091) What are the best travel destinations in Europe?


[{'rank': 1, 'score': 0.43524778, 'document': 'What is quantum computing?'},
 {'rank': 2,
  'score': 0.31878263,
  'document': 'What is the best way to learn Python?'},
 {'rank': 3, 'score': 0.11044082, 'document': 'How do I build a mobile app?'},
 {'rank': 4,
  'score': 0.09112659,
  'document': 'How do I set up a local server?'},
 {'rank': 5,
  'score': 0.090647854,
  'document': 'What are the best travel destinations in Europe?'}]

### Questions:
- What are the top-ranked results for the given queries?
- How can you improve the ranking explanation for users?
- Try this approach with a larger dataset

In [ ]:
print("The most relevant results for the programming query should be Python, local server setup, mobile app development, and computing-related documents.")  # Answers the lab question about expected top-ranked results.
print("The ranking explanation can be improved by showing each similarity score and highlighting the words or concepts that made the match strong.")  # Suggests a practical way to make semantic search explanations clearer.


The most relevant results for the programming query should be Python, local server setup, mobile app development, and computing-related documents.
The ranking explanation can be improved by showing each similarity score and highlighting the words or concepts that made the match strong.


: 